In [9]:
!rm -rf /content/drive


In [10]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [11]:
import os


BASE_DIR = "/content/drive/MyDrive/med-embed-finetune"
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)     # 학습 데이터 (query, positive 쌍)
os.makedirs(f"{BASE_DIR}/corpora", exist_ok=True)  # 검색 대상(예: ICD 요약 텍스트)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)   # 파인튜닝 결과 모델 저장

print("폴더 구조 생성 완료")



폴더 구조 생성 완료


In [1]:
!pip -q install -U sentence-transformers datasets accelerate

print("라이브러리 설치 완료")

라이브러리 설치 완료


In [12]:
import json
import os
import random

BASE_DIR = "/content/drive/MyDrive/med-embed-finetune"
os.makedirs(f"{BASE_DIR}/data", exist_ok=True)

train_path = f"{BASE_DIR}/data/train_pairs.jsonl"
val_path = f"{BASE_DIR}/data/val_pairs.jsonl"

confusing_negatives_map = {
    "소화불량": ["피로", "치통"],
    "두통": ["피로"],
    "감기": ["기침", "코막힘"],
    "피로": ["두통", "소화불량"],
    "기침": ["감기"],
    "코막힘": ["감기"]
}

def build_samples(query_list, positive_label):
    all_labels = ["소화불량", "체함", "두통", "감기", "치통", "피로", "구내염", "기침", "코막힘", "결막염", "비염"]
    confusing = confusing_negatives_map.get(positive_label, [])
    negative_pool = [n for n in all_labels if n != positive_label and n not in confusing]

    return [
        {
            "query": q,
            "positive": positive_label,
            "negative": random.sample(negative_pool, 2)
        }
        for q in query_list
    ]

# ---- 배 표현형 ----
belly_expression_train = [
    "배아파", "배가 아파요", "배가 아픈데 약 좀 추천해줘", "배가 너무 아파서 병원 가야 할까요?",
    "배가 아파서 아무것도 못 하겠어", "배가 아파서 힘들어", "배가 너무 아파서 약이라도 먹고 싶어요",
    "배가 너무 아파요. 어떻게 해야 하나요?", "배가 아픈데 무슨 약을 먹어야 하나요?",
    "배가 아파서 잠이 안 와요", "배가 아픈데 식사를 해도 괜찮을까요?", "배가 아픈데 물 마셔도 돼요?",
    "배가 너무 아파서 통증이 심해졌어", "배가 아픈데 이유를 모르겠어요", "배가 아픈 느낌이야", "배 쪽이 아픈 것 같아",
    "배가 갑자기 아파", "배가 계속 아파", "배가 아프네"
]

belly_expression_val = [
    "배가 아픈데 어떤 약이 좋을까요", "배가 너무 아픈데, 약국 가면 괜찮아질까?",
    "배가 아픈데 병원에 꼭 가야 할까요?", "배가 너무 아픈데 진통제를 먹어도 될까요?",
    "배가 아픈데 응급실 가야 할 정도인가요?", "배가 아픈데 병원 예약해야 하나요?",
    "배가 너무 아픈데 장염일 수도 있나요?"
]

belly_semantic_train = [
    "복통이 있어요", "배가 더부룩해요", "배가 짝 막힌 느낌이에요", "배에서 꾸르륵 소리가 나", "배에 통증이 있어",
    "배가 울렁거려요", "배가 찌릿해요", "배가 뒤틀리는 것 같아", "장에 가스가 찬 느낌이에요", "배가 꼬이는 느낌이에요",
    "배가 묵직해요", "배가 터질 것 같아요", "배가 묘하게 불편해요", "복부에 쥐가 나요", "장기 쪽이 아파요",
    "배에 이물감이 느껴져요", "배가 더부룩하고 뻐근해요", "배 안이 불편하고 울렁거려요",
    "장 트러블이 있는 것 같아요", "배가 찌릿찌릿 아파요", "복부가 뻐근하고 더부룩해요", "속이 막혀있는 느낌이에요",
    "속이 부글거려요", "속이 뒤집힌 느낌이에요", "속이 너무 안 좋아요", "속이 안 좋아서 구토했어요",
    "속이 이상해서 트림이 자주 나와요", "속이 안 좋아서 가스가 찬 것 같아요"
]

belly_semantic_val = [
    "배가 먹먹하고 통증이 있어요", "배에 알 수 없는 통증이 있어요", "속이 답답해서 배까지 아파요",
    "식사 후에 배가 불편해요", "장 쪽이 불쾌하게 느껴져요", "속이 안 좋아요", "속이 더부룩해요",
    "속이 울렁거려요", "속이 쓰려요", "속이 답답하고 무거워요", "속이 계속 미식거려요", "속이 탈이 난 것 같아요",
    "속이 더부룩하고 뻐근해요", "배 안이 불편하고 울렁거려요"
]

# ⬇ 하드 네거티브 수동 추가 (배아파 vs 피로/치통)
hard_negative_samples = [
    {"query": "배아파", "positive": "소화불량", "negative": ["피로", "치통"]},
    {"query": "배가 아파요", "positive": "소화불량", "negative": ["피로", "치통"]},
    {"query": "배가 너무 아파요", "positive": "소화불량", "negative": ["피로", "치통"]},
] * 5  # 반복 학습

belly_train = build_samples(belly_expression_train + belly_semantic_train, "소화불량") + hard_negative_samples
belly_val = build_samples(belly_expression_val + belly_semantic_val, "소화불량")

# ---- 일반 증상 ----
general_symptoms = {
    "두통": ["머리가 아파요", "머리가 지끈거려요", "머리가 깨질 것 같아요", "머리에 통증이 있어요", "두통이 심해요"],
    "감기": ["감기에 걸렸어요", "몸살 기운이 있어요", "감기 증상이 있어요", "몸살 증상이 있는데 약 좀 추천해줘요"],
    "치통": ["이가 아파요", "이가 욱신거려요", "치통이 심해요", "이가 깨질 듯 아파요", "이가 시려요"],
    "피로": ["너무 피곤해요", "기운이 하나도 없어요", "피로가 누적됐어요", "몸이 무겁고 피곤해요", "계속 졸려요"],
    "구내염": ["입안이 헐었어요", "구내염이 생겼어요", "입안이 따가워요", "혀가 아파요", "잇몸이 헐었어요"],
    "기침": ["기침이 계속 나요", "기침이 하루종일 나오는데 약 좀 추천해줘요"],
    "코막힘": ["코가 막혔어요", "숨쉬기가 힘들어요", "코가 꽉 막혔어요", "코가 답답해요", "코가 안 통해요"]
}

general_train = []
general_val = []

for label, queries in general_symptoms.items():
    train_qs = queries[:3]
    val_qs = queries[3:]
    general_train.extend(build_samples(train_qs, label))
    general_val.extend(build_samples(val_qs, label))

# ---- 저장 ----
with open(train_path, "w", encoding="utf-8") as f:
    for ex in belly_train + general_train:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(val_path, "w", encoding="utf-8") as f:
    for ex in belly_val + general_val:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"train_pairs.jsonl 저장 완료: {len(belly_train + general_train)}개")
print(f"val_pairs.jsonl 저장 완료: {len(belly_val + general_val)}개")


train_pairs.jsonl 저장 완료: 82개
val_pairs.jsonl 저장 완료: 32개


In [4]:
import json
import os

# --- 저장 경로 설정 ---
corpus_path = "/content/drive/MyDrive/med-embed-finetune/corpora/icd_summaries.jsonl"
os.makedirs(os.path.dirname(corpus_path), exist_ok=True)

# --- ICD 요약 텍스트 전체 리스트  ---
texts = [
    ['판콜에이내복액은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 기침)에 사용한다.', '판피린티정은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.',
     '신신파스아렉스 대형은(는) 어깨걸림 등 다양한 증상(어깨걸림, 허리통, 신경통, 류마티스)에 사용한다.', '신신파스아렉스 중형은(는) 어깨걸림 등 다양한 증상(어깨걸림, 허리통, 신경통, 류마티스)에 사용한다.',
     '제일쿨파프은(는) 어깨걸림 등 다양한 증상(어깨걸림, 허리통, 신경통, 류마티스)에 사용한다.', '닥터베아제정은(는) 소화불량 등 다양한 증상(소화불량, 식욕감퇴, 과식, 체함)에 사용한다.',
     '베아제정은(는) 소화불량 등 다양한 증상(소화불량, 식욕감퇴, 과식, 체함)에 사용한다.', '훼스탈플러스정은(는) 소화불량 등 다양한 증상(소화불량, 식욕감퇴, 과식, 체함)에 사용한다.',
     '훼스탈골드정은(는) 소화불량 등 다양한 증상(소화불량, 식욕감퇴, 과식, 체함)에 사용한다.', '타이레놀정은(는) 발열 등 다양한 증상(발열, 두통, 신경통, 근육통)에 사용한다.',
     '어린이타이레놀현탁액은(는) 발열 등 다양한 증상(발열, 두통, 신경통, 근육통)에 사용한다.', '어린이타이레놀산은(는) 발열 등 다양한 증상(발열, 두통, 신경통, 근육통)에 사용한다.',
     '치센캡슐은(는) 다리 중압감 등 다양한 증상(다리 중압감, 통증, 모세혈관 취약층에 의한 장애의 보조치료, 치질과 관련된 징후의 치료)에 사용한다.',
     '프록토세딜좌약은(는) 치핵 및 항문 주위의 부종 등 다양한 증상(치핵 및 항문 주위의 부종, 가려움, 염증, 치열 완화)에 사용한다.', '센시아정은(는) 림프 부전과 관련된 증상의 개선',
     '헤모렉스크림은(는) 치질로 인한 불쾌감 등 다양한 증상(치질로 인한 불쾌감, 작열감, 가려움 일시적 완화)에 사용한다.', '니코레트껌은(는) 흡연량을 감소시키며, 담배를 끊을 수 있도록 도움',
     '지르텍정은(는) 계절성 및 다년성 알레르기성 비염 등 다양한 증상(계절성 및 다년성 알레르기성 비염, 알레르기성 결막염, 만성 특발성 두드러기, 피부가려움증)에 사용한다.',
     '알러샷연질캡슐은(는) 계절성 및 다년성 알레르기성 비염 등 다양한 증상(계절성 및 다년성 알레르기성 비염, 알레르기성 결막염, 만성 특발성 두드러기, 피부 소양증 치료)에 사용한다.',
     '클라리틴정은(는) 알레르기성 비염, 만성 특발성 두드러기 완화', '로라타딘정은(는) 재채기 등 다양한 증상(재채기, 코막힘, 가려움, 눈의 작열감)에 사용한다.',
     '잇치페이스트치약은(는) 치은염(잇몸염) 등 다양한 증상(치은염(잇몸염), 치조(이틀)농루에 의한 여러 증상(잇몸의 발적, 부기, 출혈)에 사용한다.',
     '이가탄에프캡슐은(는) 치주(치아주위조직) 치료 후 치은염(잇몸염), 경·중등도 치주(치아 주위조직)염의 보조치료', '인사돌플러스정은(는) 치주치료 후 치은염, 경․중등도 치주염의 보조치료',
     '인사돌정은(는) 치주치료 후 치은염, 치주염의 보조치료', '뮤코로솔정은(는) 급·만성기관지염, 천식성기관지염 치료', '모드콜에스연질캡슐은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.',
     '타이레놀콜드에스정은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.', '콜대원콜드큐시럽은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.',
     '보나링에이정은(는) 멀미 등 다양한 증상(멀미, 메니에르증후군, 방사선숙취에 의한 구역·구토·어지러움 개선)에 사용한다.', '키미테패치은(는) 멀미에 의한 구역‧구토의 예방',
     '스멕타현탁액은(는) 위ㆍ십이지장과 대장질환에 관련된 통증의 완화, 급ㆍ만성 설사 완화', '삼남로페라마이드캡슐은(는) 급성설사 치료, 만성설사 치료', '로프민캡슐은(는) 급성설사 치료, 만성설사 치료',
     '로페시콘츄정은(는) 장내 가스에 기인한 복부팽만감, 경련을 수반하는 설사 증상 조절 도움', '센트룸정은(는) 비타민 및 미네랄의 보급-육체피로 등 다양한 증상(비타민 및 미네랄의 보급-육체피로, 임신, 수유기, 병중)에 사용한다.',
     '텐텐츄정은(는) 눈의 건조함의 완화 등 다양한 증상(눈의 건조함의 완화, 야맹증, 뼈, 이의 발육불량)에 사용한다.', '아로나민골드프리미엄정은(는) 신경통 등 다양한 증상(신경통, 근육통, 관절통(요통, 어깨걸림 등))에 사용한다.', '아로나민골드정은(는) 신경통 등 다양한 증상(신경통, 근육통, 관절통(요통, 어깨걸림 등))에 사용한다.', '비맥스메타정은(는) 비타민 보급 등 다양한 증상(비타민 보급, 육체피로, 수유기, 병중)에 사용한다.', '마그비스피드액은(는) 비타민 보급 등 다양한 증상(비타민 보급, 육체피로, 수유기, 병중)에 사용한다.', '벤포벨S에스정은(는) 육체피로 등 다양한 증상(육체피로, 임신/수유기, 병중/병후의 체력저하시, 발육기)에 사용한다.', '투엑스비트리플정은(는) 육체피로 등 다양한 증상(육체피로, 임신, 수유기, 병중·병후의 체력저하시)에 사용한다.', '베로카퍼포먼스발포정은(는) 육체피로 해소', '판피린큐액은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 기침)에 사용한다.', '콜대원코프큐시럽은(는) 감기의 제증상(인후통 등 다양한 증상(감기의 제증상(인후통, 기침, 가래, 오한)에 사용한다.', '콜대원키즈콜드시럽은(는) 감기의 제증상 등 다양한 증상(감기의 제증상, 발열, 두통, 관절통)에 사용한다.', '테라플루나이트타임건조시럽은(는) 감기의 제증상의 완화 고초열 및 기타 상기도 알러지에 의한 제증상의 완화', '테라플루데이타임건조시럽은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.', '겔포스엘현탁액은(는) 위산과다 등 다양한 증상(위산과다, 속쓰림, 위부불쾌감, 위부팽만감)에 사용한다.', '백초시럽플러스은(는) 식욕감퇴 등 다양한 증상(식욕감퇴, 위부팽만감, 소화불량, 과식)에 사용한다.', '카리토포텐연질캡슐은(는) 잔뇨감은 없으나 야뇨 등 다양한 증상(잔뇨감은 없으나 야뇨, 빈뇨 증상 및 요배출량이 감소한 전립샘 비대에 의한 배뇨 장애, 잔뇨감이 있고 야뇨, 빈뇨 증상 및 요배출량이 감소한 전립샘 비대에 의한 배뇨장애 치료)에 사용한다.', '오큐시스점안액은(는) 눈의 건조나 눈의 피로 해소', '프렌즈아이드롭점안액쿨하이은(는) 소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감 등 다양한 증상(소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감, 눈물 보충 (눈의 건조), 눈의 피로, 눈의 흐림 해소)에 사용한다.', '로토씨큐브아쿠아차지아이점안액은(는) 눈의 피로 등 다양한 증상(눈의 피로, 누액의 보조, 하드콘택트 렌즈 또는 소프트 콘택트 렌즈를 착용했을 때의 불쾌감, 눈의 침침함 치료)에 사용한다.', '리안점안액은(는) 영양부족으로 인한 각막과 결막의 궤양성 질환에 영양공급, 콘택츠렌즈 착용 등으로 인한 각막의 미세손상 치료', '부스코판당의정은(는) 위십이지장궤양 등 다양한 증상(위십이지장궤양, 식도경련, 유문연축, 위염)에 사용한다.', '니조랄2%액은(는) 체부백선 등 다양한 증상(체부백선, 완선, 족부백선, 피부칸디다증)에 사용한다.', '후시딘연고은(는) 포도구균 등 다양한 증상(포도구균, 연쇄구균, 코리네박테륨, 클로스트리듐 적응증: 농피증)에 사용한다.', '박트로반연고은(는) 메티실린 내성균주를 포함한 황색 포도상구균 등 다양한 증상(메티실린 내성균주를 포함한 황색 포도상구균, 기타 포도상구균, 연쇄상구균과 같은 대부분의 피부감염증의 원인균, 대장균 및 인플루엔자균과 같은 그람음성균 치료)에 사용한다.', '슈다페드정은(는) 농가진 등 다양한 증상(농가진, 모낭염, 종기증, 감염성 습진과 같은 세균성 피부 감염증 치료)에 사용한다.', '오라메디연고은(는) 만성 박리성 치은염, 미란 또는 궤양을 수반하는 난치성 구내염 및 설염 치료', '디펜쿨플라스타은(는) 퇴행성 관절염 등 다양한 증상(퇴행성 관절염, 어깨관절 주위염, 힘줄 및 힘줄윤활막염, 위팔뼈상과염)에 사용한다.', '볼타렌에멀겔은(는) 힘줄 등 다양한 증상(힘줄, 인대, 근육 및 관절의 외상성 염증, 연조직 류마티스의 국소형)에 사용한다.', '안티푸라민에스로션은(는) 퇴행성관절염 등 다양한 증상(퇴행성관절염, 어깨관절주위염, 건ㆍ건초염, 상완골 상과염)에 사용한다.', '조아팝은(는) 퇴행성관절염 등 다양한 증상(퇴행성관절염, 어깨관절주위염, 건초염, 상완골 상과염)에 사용한다.', '케펨플라스타은(는) 퇴행성관절염 등 다양한 증상(퇴행성관절염, 어깨관절주위염, 건, 건초염)에 사용한다.', '케토톱플라스타은(는) 퇴행성관절염 등 다양한 증상(퇴행성관절염, 어깨관절주위염, 건, 건초염)에 사용한다.', '록센플라스타은(는) 퇴행성관절염(골관절염) 등 다양한 증상(퇴행성관절염(골관절염), 근육통, 외상후의 종창, 통증 치료)에 사용한다.', '트라스트패취은(는) 퇴행성관절염 등 다양한 증상(퇴행성관절염, 건초염, 근육통, 골관절통)에 사용한다.', '클리어틴외용액2%은(는) 여드름 치료', '뉴베인액은(는) 하지 중압감 등 다양한 증상(하지 중압감, 통증, 하지불안, 치질과 관련된 여러 증상 치료)에 사용한다.', '비아핀에멀젼은(는) 1도 등 다양한 증상(1도, 2도 화상 및 기타 비감염성 피부상처, 방사선 치료에 의한 2차적 홍반 치료)에 사용한다.', '베나치오에프액은(는) 식욕감퇴(식욕부진) 등 다양한 증상(식욕감퇴(식욕부진), 위부팽만감, 소화불량, 과식)에 사용한다.', '까스활명수큐액은(는) 식욕감퇴(식욕부진) 등 다양한 증상(식욕감퇴(식욕부진), 위부팽만감, 소화불량, 과식)에 사용한다.', '마데카솔겔은(는) 상처치료, 피부궤양의 보조거 부분 치료', '비판텐연고은(는) 상처 등 다양한 증상(상처, 화상, 찢긴 상처, 욕창)에 사용한다.', '동아D-판테놀연고은(는) 상처 등 다양한 증상(상처, 화상, 찢긴 상처, 욕창)에 사용한다.', '마데카솔케어연고은(는) 세균에 의해 2차 감염된 피부질환의 초기 치료', '리쥬비넥스크림은(는) 궤양이 생기기 쉬운 피부 상처치료', '노스카나겔은(는) 상처 조직의 치료 후 처치', '스티모린크림은(는) 화상 등 다양한 증상(화상, 외과적 상처, 궤양성 이영양변화, 피부균열 치료)에 사용한다.', '포비돈브러쉬액은(는) 수술자의 손 및 팔의 살균소독, 수술부위의 살균소독', '성광포비딘은(는) 찢긴 상처 등 다양한 증상(찢긴 상처, 화상, 창상의 살균소독, 궤양)에 사용한다.', '포타딘연고은(는) 찢긴 상처 등 다양한 증상(찢긴 상처, 화상, 창상의 살균소독, 궤양)에 사용한다.', '베타딘세정액은(는) 수술자의 손 및 팔의 살균소독, 수술부위의 살균소독', '신신티눈밴드은(는) 티눈, 굳은살 사마귀 제거', '유한비타민C정은(는) 괴혈병의 예방과 치료,비타민 C의 요구량이 증가하는 소모성 질환 치료', '비맥스메타비정은(는) 눈의 피로 등 다양한 증상(눈의 피로, 신경통, 근육통, 관절통)에 사용한다.', '마그비스피드더블액션액은(는) 비타민 보급 등 다양한 증상(비타민 보급, 육체피로, 수유기, 병중)에 사용한다.', '콜대원노즈에스시럽은(는) 콧물 등 다양한 증상(콧물, 코막힘, 재채기, 인후통)에 사용한다.', '코메키나캡슐은(는) 코막힘 등 다양한 증상(코막힘, 콧물, 재채기, 머리무거움)에 사용한다.', '오트리빈멘톨0.1%분무제은(는) 코감기 등 다양한 증상(코감기, 코막힘, 콧물, 재채기)에 사용한다.', '코앤쿨나잘스프레이은(는) 코막힘 등 다양한 증상(코막힘, 콧물, 재채기, 머리무거움 완화)에 사용한다.', '이지엔6이브연질캡슐은(는) 두통 등 다양한 증상(두통, 치통, 발치후 통증, 인후통)에 사용한다.', '탁센레이디연질캡슐은(는) 두통 등 다양한 증상(두통, 치통, 발치후 통증, 인후통)에 사용한다.', '이지엔6이브정은(는) 두통 등 다양한 증상(두통, 치통, 발치후 통증, 인후통)에 사용한다.', '어린이부루펜시럽은(는) 관절염 등 다양한 증상(관절염, 발열, 요통, 동통)에 사용한다.', '탁센 연질캡슐은(는) 류마티양 관절염 등 다양한 증상(류마티양 관절염, 골관절염, 강직성 척추염, 건염)에 사용한다.', '바이엘아스피린정은(는) 감기로 인한 통증 및 열 증상의 완화 등 다양한 증상(감기로 인한 통증 및 열 증상의 완화, 두통, 치통, 인후통)에 사용한다.', '스트렙실트로키은(는) 인후염의 단기 증상 완화', '모트린정은(는) 감기로 인한 발열 및 동통 등 다양한 증상(감기로 인한 발열 및 동통, 요통, 월경곤란증, 수술후 동통 치료)에 사용한다.', '애드빌정은(는) 감기로 인한 발열 및 동통 등 다양한 증상(감기로 인한 발열 및 동통, 요통, 생리통, 류마티양 관절염)에 사용한다.', '둘코락스에스장용정은(는) 변비 등 다양한 증상(변비, 식욕부진(식욕감퇴), 복부팽만, 장내이상발효)에 사용한다.', '메이킨큐장용정은(는) 변비 등 다양한 증상(변비, 식욕부진, 복부팽만, 장내이상발효)에 사용한다.', '로게인액2%은(는) 남성형 탐모증 및 여성형 탐모증의 치료', '라라올라액은(는) 정신적 등 다양한 증상(정신적, 신체적 기능무력 증상의 보조요법, 회복기간 동안의 보조요법 도움)에 사용한다.', '대웅우루사연질캡슐은(는) 만성 간질환의 간기능 개선, 육체피로 해소', '멜라토닝크림은(는) 간반 등 다양한 증상(간반, 흑미증(기미), 주근깨, 노인성 검은 반점)에 사용한다.', '마그비맥스연질캡슐은(는) 육체피로 등 다양한 증상(육체피로, 임신, 수유기, 병중)에 사용한다.', '노스엣센스액은(는) 겨드랑이 등 다양한 증상(겨드랑이, 손, 발의 다한증 치료)에 사용한다.', '타이레놀8시간이알서방정은(는) 해열 및 감기 등 다양한 증상(해열 및 감기, 통증과 두통, 치통, 근육통)에 사용한다.', '게보린정은(는) 두통 등 다양한 증상(두통, 치통, 발치 후 통증, 인후통)에 사용한다.', '사리돈에이정은(는) 두통 등 다양한 증상(두통, 치통, 발치후 동통, 인후통)에 사용한다.', '가스큐액은(는) 위장관내 가스로 인한 복부 증상팽만감, 공기연하증의 개선', '프렌즈아이드롭점안액쿨은(는) 소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감 등 다양한 증상(소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감, 눈물 보충 (눈의 건조), 눈의 피로, 눈의 흐림 해소)에 사용한다.', '프렌즈아이드롭점안액순은(는) 소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감 등 다양한 증상(소프트콘택트렌즈 또는 하드콘택트렌즈를 착용하고 있을 때의 불쾌감, 눈물 보충 (눈의 건조), 눈의 피로, 눈의 흐림 해소)에 사용한다.', '뉴브이로토이엑스점안액은(는) 눈의 피로 등 다양한 증상(눈의 피로, 결막충혈, 눈병예방, 자외선 및 기타의 광선에 의한 눈염증)에 사용한다.', '젠텔정은(는) 회충 등 다양한 증상(회충, 요충, 십이지장충, 편충)에 사용한다.', '대웅알벤다졸정은(는) 회충 등 다양한 증상(회충, 요충, 십이지장충, 편충)에 사용한다.', '베타딘인후스프레이은(는) 구강내 살균소독 등 다양한 증상(구강내 살균소독, 인두염, 후두염, 구내염)에 사용한다.', '머시론정은(는) 피임', '신일세티리진정은(는) 계절성 및 다년성 알레르기성 비염 등 다양한 증상(계절성 및 다년성 알레르기성 비염, 알레르기성 결막염, 만성 특발성 두드러기, 피부가려움증 완화)에 사용한다.', '신신티눈고은(는) 티눈, 굳은살 사마귀 제거']

]

# --- JSONL 형식으로 저장 ---
with open(corpus_path, "w", encoding="utf-8") as f:
    for text in texts:
        f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")

print(f"ICD 요약 텍스트 {len(texts)}개 저장 완료: {corpus_path}")


ICD 요약 텍스트 1개 저장 완료: /content/drive/MyDrive/med-embed-finetune/corpora/icd_summaries.jsonl


In [13]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import json
import os

os.environ["WANDB_DISABLED"] = "true"


# 데이터 로딩
def load_train_data(path):
    examples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            anchor = item['query']
            positives = item['positive']
            # positive가 리스트면 하나씩 반복, 아니면 그냥 추가
            if isinstance(positives, list):
                for pos in positives:
                    examples.append(InputExample(texts=[anchor, pos]))
            else:
                examples.append(InputExample(texts=[anchor, positives]))
    return examples

# 설정
model_name = "jhgan/ko-sroberta-multitask"
train_path = "/content/drive/MyDrive/med-embed-finetune/data/train_pairs.jsonl"
val_path = "/content/drive/MyDrive/med-embed-finetune/data/val_pairs.jsonl"
save_dir = "/content/drive/MyDrive/med-embed-finetune/saved_model"

batch_size = 8
num_epochs = 3
lr = 5e-5

# 모델 불러오기
model = SentenceTransformer(model_name)

# 데이터 로딩
train_examples = load_train_data(train_path)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# 손실 함수 설정
train_loss = losses.MultipleNegativesRankingLoss(model)

# 파인튜닝
print("파인튜닝 시작")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=100,
    output_path=save_dir,
    show_progress_bar=False  # wandb 시각화 꺼짐

)
print("파인튜닝 완료. 모델 저장됨:", save_dir)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


파인튜닝 시작


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'train_runtime': 4.7208, 'train_samples_per_second': 52.11, 'train_steps_per_second': 6.99, 'train_loss': 1.6440136071407434, 'epoch': 3.0}
파인튜닝 완료. 모델 저장됨: /content/drive/MyDrive/med-embed-finetune/saved_model


In [21]:
from sentence_transformers import SentenceTransformer, util
import torch

# --- 0. 저장된 모델 로드 ---
MODEL_PATHS = {
    "파인튜닝 전": "jhgan/ko-sroberta-multitask",
    "파인튜닝 후": "/content/drive/MyDrive/med-embed-finetune/saved_model"
}

# --- 1. 증상별 테스트 쿼리 ---
test_sets = {
    "소화불량": [
        "속이 더부룩한데 괜찮아질 약 있을까요?",
        "배가 더부룩하고 더부룩한 느낌이 오래가요",
        "배에서 꾸르륵 소리 자주 나요",
        "배가 살짝 아픈데 소화제 먹어도 될까요?",
        "속이 더부룩하고 미식거려요",
        "방금 밥을 먹었는데 체한 것 같아요",
        "배아파",
        "속이 너무 안좋아요",
        "배가 너무 아파",
        "배가 너무 아픈데 약 좀 추천해줘"
    ],
    "두통": [
        "머리가 깨질 듯 아파요",
        "머리가 지끈거려요",
        "머리에 심한 통증이 있어요"
    ],
    "감기": [
        "몸살 기운이 있어요",
        "감기에 걸린 것 같아요",
        "목이 칼칼해요"
    ],
    "치통": [
        "이가 욱신거려요",
        "치통이 심해요",
        "이가 너무 아파요",
        "이빨이 너무 아파요"
    ],
    "피로": [
        "몸이 너무 피곤해요",
        "기운이 하나도 없어요",
        "피로가 누적된 것 같아요",
        "너무 피곤해서 그런데 약 좀 추천해줘요"
    ],
    "구내염": [
        "구내염 때문에 아파요",
        "혀가 따가워요",
        "입안의 점막이 붉게 부어올랐어"
    ],
    "기침": [
        "기침이 계속 나요",
        "기침이 너무 자주 나와서 힘들어요",
    ],
    "코막힘": [
        "코가 꽉 막혔어요",
        "숨쉬기가 힘들어요",
        "코가 답답해요",
        "콧물이 계속 나요"
    ]
}

# --- 2. 전체 라벨 정의 ---
all_labels = list(test_sets.keys())

# --- 3. 각 모델 평가 ---
for model_name, model_path in MODEL_PATHS.items():
    print(f"\n[{model_name}] 평가 시작")
    model = SentenceTransformer(model_path)
    label_embeddings = model.encode(all_labels, convert_to_tensor=True)

    correct = 0
    total = 0

    # 틀린 내역과 맞은 내역 로그 출력
    # for true_label, queries in test_sets.items():
    #     for query in queries:
    #         total += 1
    #         query_embedding = model.encode(query, convert_to_tensor=True)
    #         cosine_scores = util.cos_sim(query_embedding, label_embeddings)[0]
    #         top_index = torch.argmax(cosine_scores).item()
    #         predicted_label = all_labels[top_index]

    #         if predicted_label == true_label:
    #             correct += 1
    #             print(f"✔ 맞음: [{true_label}] ← \"{query}\"")
    #         else:
    #             print(f"✘ 틀림: [{true_label}] → (예측: {predicted_label}) ← \"{query}\"")


    for positive_label, queries in test_sets.items():
        for query in queries:
            total += 1
            query_embedding = model.encode(query, convert_to_tensor=True)
            cosine_scores = util.cos_sim(query_embedding, label_embeddings)[0]
            top_index = torch.argmax(cosine_scores).item()
            predicted_label = all_labels[top_index]

            if predicted_label == positive_label:
                correct += 1

    accuracy = correct / total * 100
    print(f"{model_name} 정확도: {accuracy:.2f}%")



[파인튜닝 전] 평가 시작
파인튜닝 전 정확도: 87.88%

[파인튜닝 후] 평가 시작
파인튜닝 후 정확도: 96.97%
